In [0]:
acc_name = "medallionazure"
container = "data"
acc_key = dbutils.secrets.get(scope="azure-storage", key="storagekeybdg")
spark.conf.set(f"fs.azure.account.key.{acc_name}.blob.core.windows.net", acc_key)
bronze_path = f"wasbs://{container}@{acc_name}.blob.core.windows.net/bronze/yellow_taxi"
silver_path = f"wasbs://{container}@{acc_name}.blob.core.windows.net/silver/yellow_taxi"
df_raw = spark.read.parquet(bronze_path)
print(f"Sukces! Wczytano dane z Bronze.")

In [0]:
from pyspark.sql.types import StructType, StructField, LongType, DoubleType, StringType, TimestampType
from pyspark.sql.functions import col, current_timestamp, lit

bronze_dir = f"wasbs://data@{acc_name}.blob.core.windows.net/bronze/yellow_taxi/"
silver_path = f"wasbs://data@{acc_name}.blob.core.windows.net/silver/yellow_taxi"

# 2. Pobieramy listę wszystkich plików w folderze bronze
files = [f.path for f in dbutils.fs.ls(bronze_dir) if f.name.endswith(".parquet")]
print(f"Znaleziono {len(files)} plików do przetworzenia.")

# 3. Funkcja ujednolicająca schemat dla pojedynczego pliku
def process_file(path):
    # Czytamy pojedynczy plik (tu nie będzie konfliktu schematów)
    temp_df = spark.read.parquet(path)
    
    # Wszystkie problematyczne kolumny rzutujemy na DOUBLE
    # Wybieramy tylko te, które nas interesują, żeby nie było śmieci
    cols_to_cast = [
        "passenger_count", "trip_distance", "fare_amount", "extra", 
        "mta_tax", "tip_amount", "tolls_amount", "improvement_surcharge", 
        "total_amount", "congestion_surcharge", "airport_fee"
    ]
    
    for c in cols_to_cast:
        if c in temp_df.columns:
            temp_df = temp_df.withColumn(c, col(c).cast("double"))
        else:
            temp_df = temp_df.withColumn(c, lit(0.0)) # Jeśli kolumny nie ma w starym pliku, dajemy 0.0
            
    # Upewniamy się, że ID to Long
    for c in ["PULocationID", "DOLocationID", "VendorID", "RatecodeID"]:
        if c in temp_df.columns:
            temp_df = temp_df.withColumn(c, col(c).cast("long"))
            
    return temp_df.select(
        "VendorID", "tpep_pickup_datetime", "tpep_dropoff_datetime", 
        "passenger_count", "trip_distance", "RatecodeID", 
        "PULocationID", "DOLocationID", "fare_amount", "total_amount"
    )

# 4. Łączymy wszystko w pętli (Union)
# Zaczynamy od pierwszego pliku
final_df = process_file(files[0])

# Dołączamy resztę (robimy to partiami, żeby nie przeciążyć Sparka)
for i in range(1, len(files)):
    print(f"Łączę plik {i+1}/{len(files)}...")
    next_df = process_file(files[i])
    final_df = final_df.unionByName(next_df)

# 5. Ostateczne czyszczenie i zapis
final_df.filter(col("fare_amount") > 0) \
        .filter(col("passenger_count") > 0) \
        .withColumn("processed_at", current_timestamp()) \
        .write.mode("overwrite").parquet(silver_path)

print(f"✅ BASTA! Warstwa Silver w końcu utworzona.")

In [0]:
path = f"wasbs://data@medallionazure.blob.core.windows.net/silver/yellow_taxi"
size_bytes = sum(f.size for f in dbutils.fs.ls(path) if not f.isDir())
print(f"Rozmiar warstwy Bronze: {size_bytes / (1024**3):.2f} GB")